In [ ]:
from binance.client import Client
import time

# pip install python-binance
API_KEY = "Your Api Key"
SECRET_KEY = "Your Binance Secret Key"

In [5]:
client = Client(API_KEY, SECRET_KEY, testnet=True)
client.FUTURES_URL = 'https://testnet.binancefuture.com/fapi'

In [6]:
client.futures_account()

{'feeTier': 0,
 'canTrade': True,
 'canDeposit': True,
 'canWithdraw': True,
 'feeBurn': True,
 'tradeGroupId': -1,
 'updateTime': 0,
 'multiAssetsMargin': False,
 'totalInitialMargin': '0.00000000',
 'totalMaintMargin': '0.00000000',
 'totalWalletBalance': '5000.00000000',
 'totalUnrealizedProfit': '0.00000000',
 'totalMarginBalance': '5000.00000000',
 'totalPositionInitialMargin': '0.00000000',
 'totalOpenOrderInitialMargin': '0.00000000',
 'totalCrossWalletBalance': '5000.00000000',
 'totalCrossUnPnl': '0.00000000',
 'availableBalance': '5000.00000000',
 'maxWithdrawAmount': '5000.00000000',
 'assets': [{'asset': 'FDUSD',
   'walletBalance': '0.00000000',
   'unrealizedProfit': '0.00000000',
   'marginBalance': '0.00000000',
   'maintMargin': '0.00000000',
   'initialMargin': '0.00000000',
   'positionInitialMargin': '0.00000000',
   'openOrderInitialMargin': '0.00000000',
   'maxWithdrawAmount': '0.00000000',
   'crossWalletBalance': '0.00000000',
   'crossUnPnl': '0.00000000',
   

In [7]:
print(client.API_URL)
print(client.FUTURES_URL)


https://api.binance.com/api
https://testnet.binancefuture.com/fapi


In [8]:
symbol = 'BTCUSDT'
buy_price_threshold = 67374.69
sell_price_threshold = 67375
trade_quantity = 0.002

In [9]:
# def get_current_price(symbol):
#     ticker = client.get_symbol_ticker(symbol=symbol) 
#     return float(ticker['price'])

In [10]:
def get_current_price(symbol):
    mark = client.futures_mark_price(symbol=symbol)
    return float(mark['markPrice'])

In [11]:
get_current_price(symbol)

67853.55608696

In [12]:
def place_buy_order(symbol, quantity):

    symbol = symbol.upper()

    price = float(client.futures_mark_price(symbol=symbol)['markPrice'])

    if quantity * price < 100:
        print("Increase quantity. Minimum notional is 100 USDT.")
        return

    try:
        order = client.futures_create_order(
            symbol=symbol,
            side='BUY',
            type='MARKET',
            quantity=quantity
        )
        print("Order successful:", order)

    except Exception as e:
        print("Order failed:", e)


In [13]:
place_buy_order(symbol, trade_quantity)

Order successful: {'orderId': 12410491896, 'symbol': 'BTCUSDT', 'status': 'NEW', 'clientOrderId': 'fn6RSplCg2JeQQqTd4KVhq', 'price': '0.00', 'avgPrice': '0.00', 'origQty': '0.002', 'executedQty': '0.000', 'cumQty': '0.000', 'cumQuote': '0.00000', 'timeInForce': 'GTC', 'type': 'MARKET', 'reduceOnly': False, 'closePosition': False, 'side': 'BUY', 'positionSide': 'BOTH', 'stopPrice': '0.00', 'workingType': 'CONTRACT_PRICE', 'priceProtect': False, 'origType': 'MARKET', 'priceMatch': 'NONE', 'selfTradePreventionMode': 'EXPIRE_MAKER', 'goodTillDate': 0, 'updateTime': 1771570508615}


In [14]:
def place_sell_order(symbol, quantity):
    try:
        order = client.futures_create_order(symbol=symbol, 
                                            side='SELL',
                                            type='MARKET',
                                            quantity=quantity,
                                            reduceOnly=True 
                                            )
        print("Sell order placed:", order.get('orderId'))
    except Exception as e:
        print("sell failed:", e)

In [15]:
place_sell_order(symbol, trade_quantity)

Sell order placed: 12410492135


In [16]:
# def trading_bot():
#     in_position = False
#     while True: 
#         current_price = get_current_price(symbol)
#         print(f"Current price of {symbol}: {current_price}")

#         if not in_position:
#             if current_price < buy_price_threshold:
#                 print(f"Price is below {buy_price_threshold}, Placing buy order.")
#                 place_buy_order(symbol, trade_quantity)
#                 in_position = True
#         else:
#             if current_price > sell_price_threshold:
#                 print(f"Price is above {sell_price_threshold}. Placing sell order.")
#                 place_sell_order(symbol, trade_quantity)
#                 in_position = False

#         time.sleep(3)

In [17]:
def check_position(symbol):
    positions = client.futures_position_information(symbol=symbol)
    
    for p in positions: 
        if float(p['positionAmt']) != 0:
            return True 
        
    return False

In [18]:
def trading_bot():
    while True:
        try:
            current_price = get_current_price(symbol)
            print(f"Current price: {current_price}")

            in_position = check_position(symbol)

            if not in_position and current_price < buy_price_threshold:
                print('Buying...')
                place_buy_order(symbol, trade_quantity)
                time.sleep(2)
            
            elif in_position and current_price > sell_price_threshold:
                print('Selling...')
                place_sell_order(symbol, trade_quantity)
                time.sleep(2)

            time.sleep(2)
    
        except Exception as e:
            print("Error:", e)
            time.sleep(3)


In [19]:
print("Buy threshold:", buy_price_threshold)
print("Sell threshold:", sell_price_threshold)


Buy threshold: 67374.69
Sell threshold: 67375


In [20]:
if __name__ == "__main__":
    trading_bot()

Current price: 67853.60501812
Current price: 67853.72501812
Current price: 67852.15023551
Current price: 67848.42630435


KeyboardInterrupt: 